# Building a Project with LLMs 
### -  Youtube Video Summariser
source: [freecodecamp](https://www.freecodecamp.org/news/how-to-start-building-projects-with-llms/)

In [24]:
from together import Together
from dotenv import load_dotenv
import os
load_dotenv()

api_key=os.getenv("API_KEY")
client = Together(api_key=api_key)

response = client.chat.completions.create(
    model="meta-llama/Llama-3.3-70B-Instruct-Turbo-Free",
    messages=[{"role": "user", "content": "What are some fun things to do in New York?"}],
)
print(response.choices[0].message.content)

The city that never sleeps! New York has endless options for entertainment, culture, and adventure. Here are some fun things to do in New York:

**Iconic Landmarks:**

1. Statue of Liberty and Ellis Island: Take a ferry to Liberty Island to see the iconic statue up close and visit the Ellis Island Immigration Museum.
2. Central Park: Explore the park's many walking paths, lakes, and landmarks like the Bethesda Fountain and Loeb Boathouse.
3. Times Square: Experience the bright lights and energy of the "Crossroads of the World."
4. Empire State Building: Enjoy panoramic views of the city from the observation deck on the 86th floor.
5. Brooklyn Bridge: Walk or bike across the iconic bridge for spectacular city views.

**Museums and Galleries:**

1. The Metropolitan Museum of Art: One of the world's largest and most renowned museums, with a collection that spans over 5,000 years of human history.
2. American Museum of Natural History: Explore exhibits on dinosaurs, space, and the natural 

In [3]:
## setting up the language model
from langchain_together import ChatTogether
## import api_key

llm = ChatTogether(api_key=api_key,temperature=0.0, 
                   model="meta-llama/Llama-3.3-70B-Instruct-Turbo-Free")


In [26]:
## import the youtube documnent loader from LangChain
from langchain_community.document_loaders import YoutubeLoader

video_url = 'https://www.youtube.com/watch?v=lsMQRaeKNDk'
loader = YoutubeLoader.from_youtube_url(video_url, add_video_info=False)
data = loader.load()

[![IMAGE ALT TEXT HERE](https://img.youtube.com/vi/lsMQRaeKNDk/0.jpg)](https://www.youtube.com/watch?v=lsMQRaeKNDk)

In [18]:
# show the extracted page content
data[0].page_content

'What is a REST API? What are the benefits - and how\xa0are they fundamental to your cloud application development? I\'m Nathan Heckman from IBM Cloud, and I\'m going to answer that for you today, but before i do please hit that subscribe button. Let\'s jump in with an example. Let\'s say that you work for an ice cream shop and you\'re trying to\xa0build a web application to show the flavors of ice cream that are in stock that day and allow the\xa0workers to actually make updates to those flavors. How do you do this? Well, with a REST API. You\xa0have your web app or web page communicate with a cloud-based server via a REST API. Great! So, let\'s jump into what exactly a REST API is. Starting out with: what does "REST" stand for? So, REST stands for Representational State Transfer. So it\'s "REST". Those words might not mean a\xa0whole lot to you but, basically, it\'s a standardized software architecture style, a specific type\xa0of API that\'s industry known and used. So, the first th

In [19]:
# This code creates a list of messages for the language model:
# 1. A system message with instructions on how to summarize the video transcript
# 2. A human message containing the actual video transcript

# The messages are then passed to the language model (llm) for processing
# The model's response is stored in the 'ai_msg' variable and returned

messages = [
    (
        "system", 
        """Read through the entire transcript carefully.
           Provide a concise summary of the video's main topic and purpose.
           Extract and list the five most interesting or important points from the transcript. For each point: State the key idea in a clear and concise manner.

        - Ensure your summary and key points capture the essence of the video without including unnecessary details.
        - Use clear, engaging language that is accessible to a general audience.
        - If the transcript includes any statistical data, expert opinions, or unique insights, prioritize including these in your summary or key points.""",
    ),
    ("human", data[0].page_content),
]
ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content="**Summary:** The video explains what a REST API is, its benefits, and how it's fundamental to cloud application development. REST API (Representational State Transfer) is a standardized software architecture style that enables communication between a client and a server. It's simple, scalable, and stateless, making it a popular choice for cloud applications. The video uses an example of an ice cream shop to illustrate how REST APIs work, including creating, reading, updating, and deleting resources.\n\n**Five Key Points:**\n\n1. **Definition of REST API**: REST API stands for Representational State Transfer, a standardized software architecture style that enables communication between a client and a server.\n2. **Benefits of REST APIs**: REST APIs are simple, scalable, and stateless, making them a popular choice for cloud applications. They also support caching, which improves performance.\n3. **Key Components of a REST API**: A REST API consists of a request (with a

In [20]:
# Set up a prompt template for summarizing a video transcript using LangChain

# Import necessary classes from LangChain
from langchain.prompts import PromptTemplate
from langchain import LLMChain
# from langchain.runnables import RunnableSequence
# from langchain.chains import RunnableSequence
# from langchain.llms import YourLLMClass

# Define a PromptTemplate for summarizing video transcripts
# The template includes instructions for the AI model on how to process the transcript
product_description_template = PromptTemplate(
    input_variables=["video_transcript"],
    template="""
    Read through the entire transcript carefully.
           Provide a concise summary of the video's main topic and purpose.
           Extract and list the five most interesting or important points from the transcript. 
           For each point: State the key idea in a clear and concise manner.

        - Ensure your summary and key points capture the essence of the video without including unnecessary details.
        - Use clear, engaging language that is accessible to a general audience.
        - If the transcript includes any statistical data, expert opinions, or unique insights, 
        prioritize including these in your summary or key points.

    Video transcript: {video_transcript}    """
)

In [21]:
## invoke the chain with the video transcript 

chain = LLMChain(llm=llm, prompt=product_description_template)
# chain = RunnableSequence(prompt | llm)

# Run the chain with the provided product details
summary = chain.invoke({
    "video_transcript": data[0].page_content
})

In [22]:
summary['text']

"**Summary:** The video explains what a REST API is, its benefits, and how it's used in cloud application development. REST API is a standardized software architecture style that enables communication between a client and a server. It's simple, scalable, and stateless, making it a fundamental component of cloud application development.\n\n**Five Key Points:**\n\n1. **Definition of REST API**: REST API stands for Representational State Transfer, a standardized software architecture style that enables communication between a client and a server.\n2. **Benefits of REST API**: REST APIs are simple, scalable, and stateless, making them easy to use and maintain. They also support caching, which improves performance.\n3. **Key Components of REST API**: The main building blocks of a REST API are requests and responses. Requests consist of operations (HTTP methods), endpoints, parameters, and headers, while responses are typically in JSON format.\n4. **HTTP Methods**: REST APIs use HTTP methods

In [23]:
from IPython.display import Markdown, display

display(Markdown(summary['text']))

**Summary:** The video explains what a REST API is, its benefits, and how it's used in cloud application development. REST API is a standardized software architecture style that enables communication between a client and a server. It's simple, scalable, and stateless, making it a fundamental component of cloud application development.

**Five Key Points:**

1. **Definition of REST API**: REST API stands for Representational State Transfer, a standardized software architecture style that enables communication between a client and a server.
2. **Benefits of REST API**: REST APIs are simple, scalable, and stateless, making them easy to use and maintain. They also support caching, which improves performance.
3. **Key Components of REST API**: The main building blocks of a REST API are requests and responses. Requests consist of operations (HTTP methods), endpoints, parameters, and headers, while responses are typically in JSON format.
4. **HTTP Methods**: REST APIs use HTTP methods (POST, GET, PUT, DELETE) to perform CRUD (Create, Read, Update, Delete) operations, allowing clients to interact with servers.
5. **Real-World Example**: The video uses an example of an ice cream shop to demonstrate how REST APIs work, showing how to retrieve, update, and create data using different HTTP methods and endpoints, highlighting the simplicity and effectiveness of REST APIs in real-world applications.